# ProtT5 ablation matrix and fair encoder comparison

**Status:** drafting code for review.

**When to run:** after notebook 01 establishes classical baselines and
notebook 02 fixes the evaluation protocols to be reported.

**Inputs:** complete frozen embedding caches and one selected split set.

**Outputs:** per-seed ablation metrics and predictions under
`tables/drafting_code` when enabled.

**Safety:** no test-driven model selection. Feature operator and common
dimension are specified before fitting. Pooling ablations require new
residue-level caches and are listed separately rather than faked from mean
pooled vectors. `QUICK_MODE=True` is a one-seed smoke run.

**Shared protocol:** set `SNOOPPI_ROOT` and `SNOOPPI_SPLIT_DIR` before setup.
Use the same split directory in every draft. Existing caches use original raw
sequence keys; this suite does not regenerate embeddings. Metadata uses full
sequence hashes, while leakage audits also check normalized 1,024-residue inputs.


## 1. Setup and ablation registry

In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection

from snooppi_utils import (
    build_pair_features,
    load_embedding_cache,
    select_threshold,
    summarize_predictions,
)

WRITE_OUTPUTS = False
RUN_TAG = "prott5_ablation_v1"
QUICK_MODE = True
SEEDS = [44] if QUICK_MODE else [41, 42, 43, 44, 45]

CACHE_PATHS = cache_paths(PATHS)
ENCODERS_TO_RUN = list(CACHE_PATHS)
ALLOW_PREPROCESSING_DIFFERENCES = False
MAX_TRAIN_ROWS = 5_000 if QUICK_MODE else None

PAIR_OPERATOR_ABLATIONS = {
    "sum": ("sum",),
    "absolute_difference": ("absdiff",),
    "product": ("product",),
    "sum_plus_absolute_difference": ("sum", "absdiff"),
    "sum_plus_product": ("sum", "product"),
    "absolute_difference_plus_product": ("absdiff", "product"),
    "full_symmetric": ("sum", "absdiff", "product"),
    "ordered_concat_diagnostic": ("concat",),
}
COMMON_DIMENSIONS = [256] if QUICK_MODE else [256, 512]

## 2. Load data and caches

In [ ]:
from sklearn.model_selection import train_test_split
splits = load_splits(PATHS)
assert_model_ready(splits)
manifest = split_manifest(PATHS)
print("Split hashes:", manifest["sha256"])
print(require_comparable_preprocessing(splits, ENCODERS_TO_RUN, ALLOW_PREPROCESSING_DIFFERENCES))
train_frame = splits["train"]
if MAX_TRAIN_ROWS is not None and len(train_frame) > MAX_TRAIN_ROWS:
    train_frame, _ = train_test_split(train_frame, train_size=MAX_TRAIN_ROWS,
                                    stratify=train_frame[LABEL], random_state=44)
    train_frame = train_frame.reset_index(drop=True)
y_train = train_frame[LABEL].to_numpy()
y_validation = splits["validation"][LABEL].to_numpy()
y_test = splits["test"][LABEL].to_numpy()
test_identity = prediction_identity(splits["test"], manifest)
caches, cache_records = {}, {}
for encoder_name in ENCODERS_TO_RUN:
    cache_path = CACHE_PATHS[encoder_name]
    caches[encoder_name], metadata = load_embedding_cache(cache_path, encoder_name)
    coverage = validate_cache_coverage(caches[encoder_name], splits)
    cache_records[encoder_name] = {"path": str(cache_path), "sha256": file_sha256(cache_path),
                                   "metadata": metadata, "coverage": coverage}
    print(encoder_name, coverage, metadata)
if "ProtT5" not in caches:
    raise ValueError("ProtT5 must be selected for the primary operator ablations")


## 3. Pair-operator ablations

These experiments isolate how two pooled protein embeddings are combined.
They do not answer whether ProtT5 pooling itself is superior. The ordered
concatenation diagnostic intentionally violates partner-swap invariance;
compare it with a swapped-partner test before considering it further.

In [ ]:
def fit_logistic(seed, x_train, x_validation, x_test):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=3_000,
            class_weight="balanced",
            random_state=seed,
        )),
    ])
    model.fit(x_train, y_train)
    validation_probability = model.predict_proba(x_validation)[:, 1]
    threshold = select_threshold(y_validation, validation_probability)
    test_probability = model.predict_proba(x_test)[:, 1]
    return threshold, test_probability, model


ablation_rows = []
prediction_rows = []
prot_cache = caches["ProtT5"]
for ablation_name, operations in PAIR_OPERATOR_ABLATIONS.items():
    x_train = build_pair_features(train_frame, prot_cache, operations)
    x_validation = build_pair_features(splits["validation"], prot_cache, operations)
    x_test = build_pair_features(splits["test"], prot_cache, operations)
    for seed in SEEDS:
        threshold, probability, fitted_model = fit_logistic(seed, x_train, x_validation, x_test)
        metrics = summarize_predictions(y_test, probability, threshold)
        metrics.update({
            "experiment": "pair_operator",
            "encoder": "ProtT5",
            "ablation": ablation_name,
            "seed": seed,
            "feature_dimension": x_train.shape[1],
            "quick_mode": QUICK_MODE,
        })
        metrics["split_sha256"] = manifest["sha256"]["test"]
        if operations == ("concat",):
            swapped = splits["test"].copy()
            swapped[[SEQUENCE_A, SEQUENCE_B]] = swapped[[SEQUENCE_B, SEQUENCE_A]].to_numpy()
            swap_probability = fitted_model.predict_proba(build_pair_features(swapped, prot_cache, operations))[:, 1]
            metrics["swap_mean_absolute_probability_delta"] = float(np.abs(probability - swap_probability).mean())
        ablation_rows.append(metrics)
        prediction_rows.extend({
            "experiment": "pair_operator",
            "encoder": "ProtT5",
            "ablation": ablation_name,
            "seed": seed,
            **test_identity.iloc[row_index].to_dict(),
            "threshold": threshold,
            "quick_mode": QUICK_MODE,
            "label": int(label),
            "probability": float(score),
        } for row_index, (label, score) in enumerate(zip(y_test, probability)))

## 4. Common pair-feature dimension

Gaussian random projection acts on the complete symmetric **pair feature**,
not each protein independently. It equalizes the logistic head input width,
uses training inputs only, and transforms validation/test with the same fixed
projection. This does not harmonize the cached PLM preprocessing or prove that
embedding width explains encoder differences. Record the seed and projection.


In [ ]:
for common_dimension in COMMON_DIMENSIONS:
    for encoder_name, cache in caches.items():
        base_train = build_pair_features(train_frame, cache)
        base_validation = build_pair_features(splits["validation"], cache)
        base_test = build_pair_features(splits["test"], cache)
        for seed in SEEDS:
            projector = GaussianRandomProjection(n_components=common_dimension, random_state=seed)
            projected_train = projector.fit_transform(base_train)
            projected_validation = projector.transform(base_validation)
            projected_test = projector.transform(base_test)
            threshold, probability, _ = fit_logistic(seed, projected_train, projected_validation, projected_test)
            metrics = summarize_predictions(y_test, probability, threshold)
            ablation_name = f"random_projection_{common_dimension}"
            metrics.update({
                "experiment": "common_dimension",
                "encoder": encoder_name,
                "ablation": ablation_name,
                "seed": seed,
                "feature_dimension": common_dimension,
                "quick_mode": QUICK_MODE,
                "split_sha256": manifest["sha256"]["test"],
            })
            ablation_rows.append(metrics)
            prediction_rows.extend({
                "experiment": "common_dimension",
                "encoder": encoder_name,
                "ablation": ablation_name,
                "seed": seed,
                **test_identity.iloc[row_index].to_dict(),
            "threshold": threshold,
            "quick_mode": QUICK_MODE,
                "label": int(label),
                "probability": float(score),
            } for row_index, (label, score) in enumerate(zip(y_test, probability)))

ablation_metrics = pd.DataFrame(ablation_rows)
ablation_predictions = pd.DataFrame(prediction_rows)
summary = (
    ablation_metrics.groupby(["experiment", "encoder", "ablation"])
    .agg(
        macro_ap_mean=("macro_ap", "mean"),
        macro_ap_sd=("macro_ap", "std"),
        mcc_mean=("mcc", "mean"),
        brier_mean=("brier", "mean"),
        seeds=("seed", "nunique"),
    )
    .reset_index()
    .sort_values("macro_ap_mean", ascending=False)
)
display(summary.round(4))

## 5. Visual comparison and optional outputs

In [ ]:
pair_summary = summary.loc[summary["experiment"].eq("pair_operator")].sort_values("macro_ap_mean")
figure, axis = plt.subplots(figsize=(9, max(4.5, len(pair_summary) * 0.45)))
axis.barh(pair_summary["ablation"], pair_summary["macro_ap_mean"], xerr=pair_summary["macro_ap_sd"].fillna(0), color="#176B87")
axis.set(
    title="ProtT5 pair-operator ablations",
    xlabel=("Test macro average precision (development run; one seed)" if QUICK_MODE
            else "Test macro average precision (mean ± SD across seeds)"),
    ylabel="",
    xlim=(0, 1),
)
figure.tight_layout()
plt.show()

if WRITE_OUTPUTS:
    output_files = {
        "metrics": PATHS["tables"] / f"{RUN_TAG}_metrics.csv",
        "predictions": PATHS["tables"] / f"{RUN_TAG}_test_predictions.csv",
        "summary": PATHS["tables"] / f"{RUN_TAG}_summary.csv",
        "figure": PATHS["figures"] / f"{RUN_TAG}_pair_operators.png",
    }
    output_files["manifest"] = PATHS["tables"] / f"{RUN_TAG}_manifest.json"
    prepare_outputs(output_files.values())
    save_json(make_run_manifest(PATHS, splits, {"run_tag": RUN_TAG, "quick_mode": QUICK_MODE,
              "seeds": SEEDS, "encoders": ENCODERS_TO_RUN, "operators": PAIR_OPERATOR_ABLATIONS,
              "common_pair_feature_dimensions": COMMON_DIMENSIONS, "training_rows": len(train_frame),
              "allow_preprocessing_differences": ALLOW_PREPROCESSING_DIFFERENCES}, cache_records), output_files["manifest"])
    ablation_metrics.to_csv(output_files["metrics"], index=False)
    ablation_predictions.to_csv(output_files["predictions"], index=False)
    summary.to_csv(output_files["summary"], index=False)
    figure.savefig(output_files["figure"], dpi=300, bbox_inches="tight")
    print(output_files)
else:
    print("Preview only. Use a unique RUN_TAG before saving.")

## 6. What is still required to explain “ProtT5 is best”

| Hypothesis | Required controlled ablation |
|---|---|
| Embedding width | common-dimension projection/PCA, same head and split |
| Pair operator | the symmetric operator matrix above |
| Pooling | save residue embeddings; compare mean, max, attention, and CLS-like pooling |
| Preprocessing | harmonize normalization and truncation, then rebuild every cache |
| Hyperparameters | tune each encoder on validation with the same search budget |
| Random variation | repeated seeds plus paired bootstrap confidence intervals |
| Split leakage | repeat on cold-target/cold-family protocols |
| Pretraining/task match | report model data/provenance and test across species/families |

A statistically larger score on the current pair-row test split does not
establish encoder superiority on unseen targets or families.